# 4. Analyzer custom e input per un LLM

Analizza una nota spese con uno schema custom e prepara il risultato per un LLM.

## 1. Import e configurazione

Carica librerie, variabili di ambiente, schema e documento.

In [ ]:
import json
import os
from pathlib import Path
from azure.ai.contentunderstanding import ContentUnderstandingClient, to_llm_input
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from azure.ai.contentunderstanding.models import ContentAnalyzer

In [ ]:
load_dotenv(r"..\.env", override=True)


In [ ]:
DOCUMENTO = Path(r"..\file\esercizio4.pdf")
document_bytes = DOCUMENTO.read_bytes()


In [ ]:
# Legge la definizione JSON, non il risultato di un'analisi.
definizione = json.loads(
    Path(r"..\modelli\nota_spese.json").read_text(encoding="utf-8")
)


In [ ]:
credential = DefaultAzureCredential()
client = ContentUnderstandingClient(
    endpoint=os.environ["FOUNDRY_SERVICES_ENDPOINT"],
    credential=credential,
)


In [ ]:
# Crea l'analyzer dal JSON; allow_replace aggiorna l'ID del laboratorio.
try:
    client.begin_create_analyzer(
        analyzer_id="spese_custom",
        resource=ContentAnalyzer(definizione),
        allow_replace=True,
    ).result()
except Exception:
    client.close()
    credential.close()
    raise


In [ ]:
# Analizza i byte del PDF con l'analyzer indicato e attende il risultato.
try:
    result = client.begin_analyze_binary(
        analyzer_id="spese_custom",
        binary_input=document_bytes,
    ).result()
finally:
    try:
        # Elimina l'analyzer temporaneo identificato da analyzer_id.
        client.delete_analyzer(analyzer_id="spese_custom")
    finally:
        client.close()
        credential.close()


## 3. Risultato

Mostra i campi estratti dalla nota spese.

In [ ]:
for content in result.contents:
    fields = content.as_dict()["fields"]
    print(json.dumps(fields, ensure_ascii=False, indent=2))

## 4. Contesto per un LLM

Combina campi e Markdown in un testo pronto per il modello.

In [ ]:
# Converte campi e Markdown del risultato in un unico testo per un LLM.
contesto = to_llm_input(result)
print(contesto)


In [ ]:
print("SOLO CAMPI")
print(to_llm_input(result, include_markdown=False))
print("\nSOLO MARKDOWN")
print(to_llm_input(result, include_fields=False))
